# 09 - Deep learning model 2: Wide & Deep (PyTorch)
Wide part = one learned weight per category; deep part = embeddings + residual MLP. Logits are added.
Needs notebook 05 output and `pip install torch`.

In [1]:
import sys, os
from pathlib import Path
# works whether Jupyter starts in the project root or inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))      # lets us import data_utils.py from the root
print("Project root:", ROOT)

Project root: c:\Users\aru12\OneDrive\Desktop\capstone project


## 1. Imports, data and category vocabularies (train only)

In [2]:
import copy, numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score
from data_utils import *

torch.manual_seed(SEED); np.random.seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
(Xtr, ytr), (Xva, yva), (Xte, yte) = load_splits()

# vocabularies come from TRAIN only; unseen categories map to index 0
vocab = {c: {v: i + 1 for i, v in enumerate(sorted(Xtr[c].unique()))} for c in FEATURES}
def enc(X):
    return torch.tensor(np.stack([X[c].map(vocab[c]).fillna(0).astype(int).to_numpy()
                                  for c in FEATURES], 1), dtype=torch.long).to(dev)
Ctr, Cva, Cte = enc(Xtr), enc(Xva), enc(Xte)
Ytr = torch.tensor(ytr, dtype=torch.float32).to(dev)

device: cpu


## 2. Model, loss and optimizer

In [3]:
class ResBlock(nn.Module):
    def __init__(self, d, p=0.2):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(p), nn.Linear(d, d))
    def forward(self, x):
        return x + self.f(x)

class WideDeep(nn.Module):
    def __init__(self):
        super().__init__()
        n = [len(vocab[c]) + 1 for c in FEATURES]
        self.wide = nn.ModuleList([nn.Embedding(k, 1) for k in n])
        self.embs = nn.ModuleList([nn.Embedding(k, min(48, k // 2 + 2)) for k in n])
        d_in = sum(e.embedding_dim for e in self.embs)
        self.inp = nn.Linear(d_in, 128)
        self.blocks = nn.Sequential(ResBlock(128), ResBlock(128))
        self.out = nn.Sequential(nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, c):
        wide = sum(w(c[:, i]) for i, w in enumerate(self.wide)).squeeze(1)
        deep = torch.cat([e(c[:, i]) for i, e in enumerate(self.embs)], 1)
        return wide + self.out(self.blocks(self.inp(deep))).squeeze(1)

net = WideDeep().to(dev)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor((ytr == 0).sum() / (ytr == 1).sum(),
                                                        dtype=torch.float32).to(dev))
opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=1)
print(net)

WideDeep(
  (wide): ModuleList(
    (0): Embedding(32, 1)
    (1): Embedding(139, 1)
    (2): Embedding(276, 1)
  )
  (embs): ModuleList(
    (0): Embedding(32, 18)
    (1): Embedding(139, 48)
    (2): Embedding(276, 48)
  )
  (inp): Linear(in_features=114, out_features=128, bias=True)
  (blocks): Sequential(
    (0): ResBlock(
      (f): Sequential(
        (0): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=128, out_features=128, bias=True)
      )
    )
    (1): ResBlock(
      (f): Sequential(
        (0): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (1): ReLU()
        (2): Dropout(p=0.2, inplace=False)
        (3): Linear(in_features=128, out_features=128, bias=True)
      )
    )
  )
  (out): Sequential(
    (0): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_run

## 3. Train (LR schedule and early stopping use validation AUC only)

In [4]:
def predict(C):
    net.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(C), 8192):
            out.append(torch.sigmoid(net(C[i:i + 8192])))
    return torch.cat(out).cpu().numpy()

best_auc, best_state, bad, BS = 0, None, 0, 2048
for ep in range(30):
    net.train(); perm = torch.randperm(len(Ctr), device=dev)
    for i in range(0, len(perm), BS):
        idx = perm[i:i + BS]
        opt.zero_grad(); loss_fn(net(Ctr[idx]), Ytr[idx]).backward(); opt.step()
    auc = roc_auc_score(yva, predict(Cva))            # early stopping uses VALIDATION only
    sched.step(auc)
    print(f"epoch {ep+1:02d}  val_auc={auc:.4f}")
    if auc > best_auc: best_auc, best_state, bad = auc, copy.deepcopy(net.state_dict()), 0
    else:
        bad += 1
        if bad >= 4: break

epoch 01  val_auc=0.9135
epoch 02  val_auc=0.9158
epoch 03  val_auc=0.9167
epoch 04  val_auc=0.9163
epoch 05  val_auc=0.9175
epoch 06  val_auc=0.9171
epoch 07  val_auc=0.9171
epoch 08  val_auc=0.9178
epoch 09  val_auc=0.9175
epoch 10  val_auc=0.9178
epoch 11  val_auc=0.9179
epoch 12  val_auc=0.9177
epoch 13  val_auc=0.9180
epoch 14  val_auc=0.9179
epoch 15  val_auc=0.9179
epoch 16  val_auc=0.9179
epoch 17  val_auc=0.9179


## 4. Evaluate on test (threshold from validation) and save the result

In [5]:
net.load_state_dict(best_state)
t = best_threshold(yva, predict(Cva))
p_te = predict(Cte)
report("Wide&Deep (DL)", yte, p_te, t)
print("test ROC-AUC:", round(roc_auc_score(yte, p_te), 4))
save_rows([metrics_row("Wide&Deep (DL)", yte, p_te, t)])
pd.read_csv(RESULTS)

Wide&Deep (DL)         thr=0.58  accuracy=0.8332  balanced_acc=0.8320  gap=0.0012
test ROC-AUC: 0.9127


,Model,Features,Train/Val/Test,Test_rows,Threshold,Accuracy,Balanced_Accuracy,Precision,Recall,F1,ROC_AUC
0,LogisticRegression,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.53,0.8271,0.8276,0.5299,0.8285,0.6463,0.9094
1,RandomForest,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.51,0.8304,0.8301,0.5357,0.8295,0.6510,0.9104
2,HistGradientBoosting,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8296,0.8299,0.5342,0.8303,0.6501,0.9113
3,XGBoost,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.52,0.8312,0.8311,0.5371,0.8309,0.6524,0.9125
4,LightGBM,Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.55,0.8302,0.8301,0.5353,0.8299,0.6508,0.9114
5,MLP (embeddings),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.57,0.8327,0.8336,0.5397,0.8350,0.6556,0.9136
6,Wide&Deep (DL),Primary Type + Location Description + Beat,2023-Jun24 / Jul-Dec24 / 2025,236927,0.58,0.8332,0.8320,0.5408,0.8301,0.6549,0.9127
